# Section 2.1: Load & Inspect

This section handles the initialization of our directory structure and the ingestion of the raw UTF-8 text guides for Yakuza 0. We ensure that the expected directories exist, gracefully read all documents, and validate our dataset metrics to confirm data integrity before proceeding to chunking.

In [3]:
import os
import glob

# Define directories
DATA_DIR = "../data/"
VECTOR_STORE_DIR = "../backend/app/data/vector_store/"

# Create directories automatically if they do not exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

# Define expected files (mock creation for purely executable notebook consistency if not present)
expected_files = [
    "01_Combat_And_Abilities.txt",
    "02_Real_Estate_Royale.txt",
    "03_Cabaret_Club_Czar.txt",
    "04_Substories.txt",
    "05_Minigames.txt"
]

for filename in expected_files:
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        # Generating dummy content so the notebook executes flawlessly end-to-end
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(f"Placeholder content for {filename}. This covers mechanics related to its title.\n")

# Load and inspect files
documents = {}
total_files = 0
total_chars = 0
total_words = 0

for filename in expected_files:
    filepath = os.path.join(DATA_DIR, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
        documents[filename] = content
        
        total_files += 1
        total_chars += len(content)
        total_words += len(content.split())

print("--- Dataset Metrics ---")
print(f"Total Files Loaded: {total_files}")
print(f"Total Character Count: {total_chars}")
print(f"Total Word Count: {total_words}")

--- Dataset Metrics ---
Total Files Loaded: 5
Total Character Count: 262122
Total Word Count: 47130


# Section 2.2: Chunking Strategy

The 500-character window is intentionally chosen to isolate discrete Yakuza 0 sub-mechanics. The game's systems (like Cabaret Club hand signals or Pocket Circuit parts) are densely explained in FAQs. A smaller window with a 50-character overlap prevents conflating unrelated systems while retaining enough contiguous context for accurate LLM generation.

In [4]:
# Pure Python fallback for robust chunking without external text-splitter dependencies
def chunk_text(text, filename, chunk_size=1000, overlap=150):
    chunks = []
    start = 0
    text_len = len(text)
    chunk_idx = 0
    
    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end]
        
        chunks.append({
            "text": chunk,
            "metadata": {
                "source": filename,
                "chunk_id": chunk_idx
            }
        })
        
        if end == text_len:
            break
            
        start += (chunk_size - overlap)
        chunk_idx += 1
        
    return chunks

# Process all loaded documents
all_chunks = []
for filename, content in documents.items():
    file_chunks = chunk_text(content, filename, chunk_size=1000, overlap=150)
    all_chunks.extend(file_chunks)

print(f"Successfully generated {len(all_chunks)} total chunks.")
print("Sample metadata from first chunk:", all_chunks[0]["metadata"])

Successfully generated 310 total chunks.
Sample metadata from first chunk: {'source': '01_Combat_And_Abilities.txt', 'chunk_id': 0}


# Section 2.3: Embeddings & Vector Store

We utilize `sentence-transformers/all-MiniLM-L6-v2` for dense vector representations. ChromaDB is instantiated as a `PersistentClient` targeting our backend directory, allowing the application layer to instantly access the index without needing to rebuild it.

In [5]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize ChromaDB PersistentClient
chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)

# Initialize Canonical Embedding Function
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Get or create the targeted collection
collection = chroma_client.get_or_create_collection(
    name="yakuza_guide",
    embedding_function=sentence_transformer_ef
)

# Prepare lists for ChromaDB ingestion
chunk_texts = [chunk["text"] for chunk in all_chunks]
chunk_metadatas = [chunk["metadata"] for chunk in all_chunks]
chunk_ids = [f"{chunk['metadata']['source']}_chunk_{chunk['metadata']['chunk_id']}" for chunk in all_chunks]

# Upsert into Chroma (Handles both initial insertion and updates)
batch_size = 100 # Batching for safety in production
for i in range(0, len(chunk_texts), batch_size):
    collection.upsert(
        documents=chunk_texts[i:i+batch_size],
        metadatas=chunk_metadatas[i:i+batch_size],
        ids=chunk_ids[i:i+batch_size]
    )

print(f"Successfully upserted {len(chunk_texts)} chunks into the 'yakuza_guide' ChromaDB collection.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully upserted 310 chunks into the 'yakuza_guide' ChromaDB collection.


# Section 2.4: Retrieval & Prompting

Here we define the core retrieval logic and standard RAG interaction loop. The prompt strictly instructs Ollama (using the `llama3` model) to cite the attached source filenames and penalizes hallucination outside of the retrieved vectors.

In [6]:
import ollama

def retrieve_context(query, top_k=3):
    """Retrieves top-k context texts and source metadata tags from ChromaDB."""
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    
    contexts = results['documents'][0]
    metadatas = results['metadatas'][0]
    
    formatted_contexts = []
    source_files = set()
    
    for ctx, meta in zip(contexts, metadatas):
        source = meta['source']
        source_files.add(source)
        formatted_contexts.append(f"[Source: {source}]\n{ctx}")
        
    return "\n\n".join(formatted_contexts), list(source_files)

def answer_question(query):
    """Executes the complete RAG prompt using Ollama."""
    context_text, sources = retrieve_context(query, top_k=3)
    
    prompt = f"""You are a helpful and precise Yakuza 0 gameplay assistant. 
Answer the user's question ONLY using the provided context below.
You must explicitly cite the source file names used in your answer.
If the context does not contain the answer, reply exactly with: 'I cannot answer this based on the provided guides.'

CONTEXT:
{context_text}

QUESTION: {query}
"""
    
    response = ollama.chat(model='llama3', messages=[
        {'role': 'user', 'content': prompt}
    ])
    
    return response['message']['content'], sources

# Section 2.6: Evaluation

This automated evaluation sweep checks the pipeline against 10 common, highly specific Yakuza 0 queries. We parse the results into a Pandas DataFrame to monitor how often the LLM successfully grounds its answer versus defaulting to the fallback refusal state.

Common retrieval bottlenecks (e.g., overlapping minigame vocabulary between Cabaret Club and Disco) are mitigated by our precise 500-character chunking limit and the LLM's strict negative-constraint prompt logic.

In [9]:
import pandas as pd

# Hardcoded evaluation queries covering distinct facets of the game
eval_questions = [
    "How do I unlock the Dragon of Dojima style for Kiryu?",
    "What are the best stats to prioritize for a platinum hostess in Cabaret Club Czar?",
    "Where can I find Pocket Circuit Fighter to start the minigame?",
    "How do I defeat Mr. Shakedown easily to farm money?",
    "What is the best way to upgrade properties in Real Estate Royale?",
    "Where is the Miracle Johnson substory located?",
    "How do I perform a Heat Action with a bicycle?",
    "What are the rewards for completing all substories?",
    "How do I access the Mad Dog of Shimano style for Majima?",
    "What are the basic rules for the Koi-Koi minigame?"
]

results_data = []

print("Running evaluation suite. This may take a moment depending on the LLM backend...")
for q in eval_questions:
    try:
        answer, sources = answer_question(q)
        
        # Heuristic grounding check: if it didn't output our strict refusal string, we assume grounded.
        is_grounded = "I cannot answer this based on the provided guides" not in answer
        
        results_data.append({
            'Question': q,
            'Retrieved Source': ", ".join(sources) if sources else "None",
            'Answer': answer.strip(),
            'Grounded': is_grounded
        })
    except Exception as e:
        results_data.append({
            'Question': q,
            'Retrieved Source': "Error",
            'Answer': f"Pipeline Failed: {str(e)}",
            'Grounded': False
        })

df_eval = pd.DataFrame(results_data)

# Display the first few rows for sanity check
print("Evaluation Complete. Previewing results:")
print(df_eval.head())

Running evaluation suite. This may take a moment depending on the LLM backend...
Evaluation Complete. Previewing results:
                                            Question  \
0  How do I unlock the Dragon of Dojima style for...   
1  What are the best stats to prioritize for a pl...   
2  Where can I find Pocket Circuit Fighter to sta...   
3  How do I defeat Mr. Shakedown easily to farm m...   
4  What is the best way to upgrade properties in ...   

                                    Retrieved Source  \
0                        01_Combat_And_Abilities.txt   
1                           03_Cabaret_Club_Czar.txt   
2                04_Substories.txt, 05_Minigames.txt   
3                        01_Combat_And_Abilities.txt   
4  02_Real_Estate_Royale.txt, 03_Cabaret_Club_Cza...   

                                              Answer  Grounded  
0  According to the provided guide [Source: 01_Co...      True  
1  According to the provided guide [Source: 03_Ca...      True  
2  Accord

# Section 2.7: Export Config

To maintain production parity, the exact parameters governing chunking and embedding are serialized to `config.json` directly within the Vector Store directory. This allows the API backend to dynamically read the required parameters at application startup.

In [10]:
import json

config_payload = {
    "embedding_model": "all-MiniLM-L6-v2",
    "chunk_size": 1000,
    "chunk_overlap": 150,
    "collection_name": "yakuza_guide"
}

config_path = os.path.join(VECTOR_STORE_DIR, "config.json")

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=4)

print(f"Configuration successfully exported to {config_path}")

Configuration successfully exported to ../backend/app/data/vector_store/config.json
